# Introduction


## The Dataset
The dataset used is from the Veneto Tumour Registry created in 2010. Veneto is a region in northern Italy (capital Venice) with a population of just under 5 million people. More details about this registry are available at https://www.registrotumoriveneto.it/english/introduction/  
This registry is part of the Association of Italian Tumour Registries (AIRTUM) which was established in 1996 to promote high quality standardised cancer registry data which is used by amongst others the Italian Ministry of Heealth. This is consdidered a gold standard in Europe. More details can be found at https://www.registri-tumori.it/cms/

The registry is of neoplastic disease, that is to say diseases involving the uncontrolled growth of cells, which when malignant lead to cancer. 


Study population: all incident cases of invasive cutaneous malignant melanoma (CMM) recorded in the Veneto Cancer Registry (Registro Tumori del Veneto – RTV) in 2015 (1,050 cases) and 2017 (1,205 cases) for which the number of mitoses was available.





In [279]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import io
import requests
import matplotlib.pyplot as plt
# seaborn is a better tool for this with more examples 
import seaborn as sns


df_melanoma = pd.read_csv('melanoma_mitoses.csv', sep=';')

In [280]:
df_melanoma.head()

,ID_prog,age,sex,year,histology,site,pT,pN,pM,TNM_stage,breslow,clark,ulcerazione,regressione,til,growth,positive_slnb,mitoses,status,survival
0,1,31,F,2015,Nodular melanoma,Trunk,pT4,N3c,M0,III,>=4,IV,Present,Absent,Absent,Vertical,0,3,0,648
1,2,53,M,2017,Superficial spreading melanoma,Trunk,pT1,N0,M0,I,< 0.76,II,Absent,Present,Present,Vertical,0,0,0,1147
2,3,67,F,2015,Superficial spreading melanoma,Upper limb,pT1,N0,M0,I,0.76-1.50,IV,Absent,Absent,Present,Vertical,0,0,0,1948
3,4,29,F,2015,Superficial spreading melanoma,Trunk,pT1,N1a,M0,III,< 0.76,III,Absent,Absent,Present,Vertical,1,0,0,2170
4,5,57,F,2015,Superficial spreading melanoma,Lower limb,pT1,N0,M0,I,< 0.76,II,Absent,Absent,Present,Radial,0,0,0,1960


In [281]:
df_melanoma.shape

(2255, 20)

In [282]:
df_melanoma.columns

Index(['ID_prog', 'age', 'sex', 'year', 'histology', 'site', 'pT', 'pN', 'pM',
       'TNM_stage', 'breslow', 'clark', 'ulcerazione', 'regressione', 'til',
       'growth', 'positive_slnb', 'mitoses', 'status', 'survival'],
      dtype='object')

In [283]:
df_melanoma[['survival', 'status']]

,survival,status
0,648,0
1,1147,0
2,1948,0
3,2170,0
4,1960,0
...,...,...
2250,1275,0
2251,436,1
2252,1297,0
2253,18,1


## Encoding the data

Logistic Regression works on numeric data, we can use the optimum encoding for the different data we have. We will also drop redundant columns.

First of all we will drop the features survival, ID_prg, and year for the following reasons...

    
| Column | Explaination |
| ------ | ------------ |
| survival | Leakage: another way of showing status the variable we are trying to predict. We choose 'status' instead as it is a classification not a continious variable. |
| ID_prog | Row identifier, no predictive value. |
| year | Although this is year of diagnosis, which is another way of showing survival (i.e. leakage) and furthermore it is not a clinical signal. |


In [284]:
# First drop unnecessary columns
df_melanoma_clean = df_melanoma.drop(columns=[ 'survival',   # leakage: another way of showing status the variable we are trying to predict. We choose 'status' instead 
                                               'ID_prog',    # Row identifier, no predictive value
                                               'year'])      # Although this is year of diagnosis, which is another way of showing survival and furthermore it is not a clinical signal.


# ORDINAL ENCODINGS 
# This for the variables where we care about the order
# so e.g. the larger the breslow_map (thicker the tumor) 
# the more dangerous it is
tnm_map = {'I': 1, 'II': 2, 'III': 3, 'IV': 4}
df_melanoma_clean['TNM_stage'] = df_melanoma_clean['TNM_stage'].map(tnm_map)

pt_map = {'pT1': 1, 'pT2': 2, 'pT3': 3, 'pT4': 4}
df_melanoma_clean['pT'] = df_melanoma_clean['pT'].map(pt_map)

breslow_map = {'< 0.76': 1, '0.76-1.50': 2, '1.51-3.99': 3, '>=4': 4}
df_melanoma_clean['breslow'] = df_melanoma_clean['breslow'].map(breslow_map)

clark_map = {'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5}
df_melanoma_clean['clark'] = df_melanoma_clean['clark'].map(clark_map)

pn_map = {'N0': 0, 'N1a': 1, 'N1b': 1, 'N1c': 1,
           'N2a': 2, 'N2b': 2, 'N2c': 2,
           'N3':  3, 'N3c': 3}
df_melanoma_clean['pN'] = df_melanoma_clean['pN'].map(pn_map)


# BINARY ENCODINGS
# Simple categorization between 2 options
df_melanoma_clean['sex']         = df_melanoma_clean['sex'].map({'F': 0, 'M': 1})
df_melanoma_clean['ulcerazione'] = df_melanoma_clean['ulcerazione'].map({'Absent': 0, 'Present': 1})
df_melanoma_clean['til']         = df_melanoma_clean['til'].map({'Absent': 0, 'Present': 1})
df_melanoma_clean['growth']      = df_melanoma_clean['growth'].map({'Radial': 0, 'Vertical': 1})
df_melanoma_clean['regressione'] = df_melanoma_clean['regressione'].map({'Absent': 0, 'Present': 1})
df_melanoma_clean['pM']          = df_melanoma_clean['pM'].map({'M0': 0, 'M1': 1})
df_melanoma_clean['positive_slnb'] = df_melanoma_clean['positive_slnb'].map({'0': 0, '1': 1, '>1': 2})


# ONE-HOT ENCODE 
# Columns we need to encode but don't care about the order
# This is a standard method of working with this kind of data
# drop_first=True drops one dummy per group to avoid multicollinearity 
# (i.e. you can determine this columns value from the state of the others)
df_melanoma_clean = pd.get_dummies(df_melanoma_clean, columns=['histology', 'site'], drop_first=True)

# LOG-TRANSFORM 
# This is the recommended way where you have a wide range of values 
# mitosis runs from 0 to 55 and is very skewed to the lower end of 
# that range
df_melanoma_clean['mitoses_log'] = np.log1p(df_melanoma_clean['mitoses'])

# ... and drop the original mitosis column
df_melanoma_clean = df_melanoma_clean.drop(columns=['mitoses'])

In [285]:
df_melanoma_clean.columns

Index(['age', 'sex', 'pT', 'pN', 'pM', 'TNM_stage', 'breslow', 'clark',
       'ulcerazione', 'regressione', 'til', 'growth', 'positive_slnb',
       'status', 'histology_Desmoplastic melanoma',
       'histology_Lentigo maligna', 'histology_Malignant melanoma',
       'histology_Nodular melanoma', 'histology_Spitzoid melanoma',
       'histology_Superficial spreading melanoma', 'site_Head',
       'site_Lower limb', 'site_Trunk', 'site_Upper limb', 'mitoses_log'],
      dtype='object')

In [286]:
df_melanoma_clean.shape

(2255, 25)

## Data Problems fitting the model.

Logistic regression does not accept NULLS or NaN values, and errors when we try to "fit" the model so we need to do some analysis to clean up. First of all let's try and get a handle on the problem...

In [287]:
df_melanoma_clean.isnull().sum()


age                                           0
sex                                           0
pT                                            3
pN                                            8
pM                                           13
TNM_stage                                    16
breslow                                       4
clark                                       206
ulcerazione                                  25
regressione                                 555
til                                         143
growth                                      390
positive_slnb                                 0
status                                        0
histology_Desmoplastic melanoma               0
histology_Lentigo maligna                     0
histology_Malignant melanoma                  0
histology_Nodular melanoma                    0
histology_Spitzoid melanoma                   0
histology_Superficial spreading melanoma      0
site_Head                               

## Missing values analysis
In Data Science there are different classifications of missing data..

| Classification | Definition |
| -------------- | ---------- |
| Missing Completely at Random (MCAR) | The distribution of missing values has no relationship to anything |
| Missing at Random (MAR) | The missing values depends on other observed variables |
| Missing Not at Random (MNAR) | the nature of the missing variables is unknown |

Thinking in these terms will allow us to make some assumptions about the data. E.g. Clark level might be missing more often for thicker tumours because clinicians didn't bother recording it when Breslow was clearly the more relevant measure. This is an example of MAR, so it is reasonable to remove the column.

Beyond that however we can not draw conclusions about the other columns given a cliniciam may have not thought them important enough to record, or may have mistakenly not recoreded anything instead of positivley affirming absence of a characteristic.

One method to compensate for missing date is imputation. "...imputation is the process of replacing missing data with substituted values" https://en.wikipedia.org/wiki/Imputation_(statistics) and is often used to fill in missing data. Simple imputation is rule based e.g. taking the mean average of the variable, whilst Multiple Imputation relies on generating multiple copies of the data. Neither of these methods are appropriate in our case as we don't have a clear expectation that a missing value is related to other data recorded elsewhere on the data row for the patient. Hence we cannot establish the case for missing data being MAR, and imputation is not suitable (https://pmc.ncbi.nlm.nih.gov/articles/PMC6293424/)

The other option open to us is simply "remove" the missing data, i.e. delete the affected rows, or in some cases a column.

### Which data to remove

Looking at the data above we can see the 3 main offenders are regressione (regression), growth and clark.Clark is an easy one discount as it is an alternate and less accurate measure of breslow (both refer to the thickness of the tumour). Regression and growth are less easy to discount but the number of rows affected makes deleting the rows impractical as it would severly reduce the size of the dataset. Therefore we will drop these columns. 


In [288]:
# Based on that let's clean up some more
df_melanoma_clean = df_melanoma_clean.drop(columns=[ 'clark',      # this is an alternative measure of tumour depth similiar to breslow. Data has many nulls so will use breslow instead
                                                     'regressione', # there are over 500 nulls in 1804 rows in the testing data. Regression in patients is independent so imputation isn't valid
                                                     'growth'])      # same argument as for regression
                                    

In [289]:
df_melanoma_clean.columns

Index(['age', 'sex', 'pT', 'pN', 'pM', 'TNM_stage', 'breslow', 'ulcerazione',
       'til', 'positive_slnb', 'status', 'histology_Desmoplastic melanoma',
       'histology_Lentigo maligna', 'histology_Malignant melanoma',
       'histology_Nodular melanoma', 'histology_Spitzoid melanoma',
       'histology_Superficial spreading melanoma', 'site_Head',
       'site_Lower limb', 'site_Trunk', 'site_Upper limb', 'mitoses_log'],
      dtype='object')


Now we can rerun the check to see gow many rows we'd lose by dropping all rows that contain nulls.

In [290]:
df_melanoma_clean.isnull().sum()


age                                           0
sex                                           0
pT                                            3
pN                                            8
pM                                           13
TNM_stage                                    16
breslow                                       4
ulcerazione                                  25
til                                         143
positive_slnb                                 0
status                                        0
histology_Desmoplastic melanoma               0
histology_Lentigo maligna                     0
histology_Malignant melanoma                  0
histology_Nodular melanoma                    0
histology_Spitzoid melanoma                   0
histology_Superficial spreading melanoma      0
site_Head                                     0
site_Lower limb                               0
site_Trunk                                    0
site_Upper limb                         

In [291]:
# Check the size of the dataframe
df_melanoma_clean.shape

(2255, 22)

In [292]:
# drop rows that contain a null value and recheck size
df_melanoma_clean = df_melanoma_clean.dropna()
df_melanoma_clean.shape

(2079, 22)

In [293]:
df_melanoma_clean['status'].value_counts()

status
0    1821
1     258
Name: count, dtype: int64

In [294]:
# check we have no nulls here
df_melanoma_clean.isnull().sum()

age                                         0
sex                                         0
pT                                          0
pN                                          0
pM                                          0
TNM_stage                                   0
breslow                                     0
ulcerazione                                 0
til                                         0
positive_slnb                               0
status                                      0
histology_Desmoplastic melanoma             0
histology_Lentigo maligna                   0
histology_Malignant melanoma                0
histology_Nodular melanoma                  0
histology_Spitzoid melanoma                 0
histology_Superficial spreading melanoma    0
site_Head                                   0
site_Lower limb                             0
site_Trunk                                  0
site_Upper limb                             0
mitoses_log                       

👍No NULL values


## Next Train the Model

In [295]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# Prepare data status is the column we are testing for

X = df_melanoma_clean.drop(columns=['status'])

# y is a series (aka a list) for target variable.
y = df_melanoma_clean['status']

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

Training set size: 1663
Test set size: 416


In [297]:
# Fit logistic regression model
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [298]:
# Make predictions on test set
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # Probability of class 1

# Evaluate performance
print("Classification Report on Test Set:")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Survived', 'Dead']))

Classification Report on Test Set:
              precision    recall  f1-score   support

    Survived       0.92      0.96      0.94       359
        Dead       0.68      0.49      0.57        57

    accuracy                           0.90       416
   macro avg       0.80      0.73      0.76       416
weighted avg       0.89      0.90      0.89       416



## Results

### Definitions of the Classification Report

| Classification | Definition | Comments |
| -------------- | ---------- | -------- |
| **Precision** | For everything the model *predicted* as that class, how many were actually correct|   "Died": of all the patients the model flagged as deaths, 68% genuinely died. The other 32% were false alarms (survivors incorrectly labelled as deaths). |
| **Recall** | Everything that *actually is* that class, how many did the model catch? | Of the 57 real deaths in the test set, the model only found 49% of them. Missing more than half from a clinical perspective is dangerous those are patients who died but the model said they'd survive, thus misleading the clinician. |
| **F1-score** | Harmonic mean of precision and recall. It penalises you if either one is poor. | The "Died" F1 of 0.57 suggests the model is struggling with the "Died" grouping, as this is much lower than the score for "Survived" |
| **Support** | Actual count of that class in your test set. | In the test dataset there are 359 survivors and 57 deaths. |

---

The model is behaving poorly most probably due to the imbalance in the data between the deaths and the survivors. Having a low "F1" score can often be indicative of this. In the original "clean" dataframe we had 1821 survivors and 258 deaths. This is the imbalance.
This is backed up by the final section of the Classification Report...

---


**Accuracy** — overall, what fraction of all 416 predictions were correct. In this case 90% is the misleading number for your dataset as a model that predicted "Survived" for every single patient would score 359/416 = 86% accuracy 

**Macro average** — averages precision, recall, and F1 across both classes equally (i.e. is unweighted average), regardless of how many patients are in each class. This treats "Died" and "Survived" as equally important. Your macro recall of 0.73 is telling you the real story — averaged fairly across both classes, the model is nowhere near as good as 90% suggests.

**Weighted average** — averages the same metrics but weights each class by its support count. Because "Survived" (359 cases) dominates, the weighted average is pulled strongly toward those numbers and flatters the model. This is why it sits close to the accuracy figure.

---


The standard way of fixing this is to use the imbalance value to allow the model to weigh correctly the imbalance


In [299]:
# Fit logistic regression model
model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_train, y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [300]:
# Make predictions on test set
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # Probability of class 1

# Evaluate performance
print("Classification Report on Test Set:")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Survived', 'Dead']))

Classification Report on Test Set:
              precision    recall  f1-score   support

    Survived       0.97      0.77      0.86       359
        Dead       0.37      0.86      0.52        57

    accuracy                           0.78       416
   macro avg       0.67      0.81      0.69       416
weighted avg       0.89      0.78      0.81       416



This is better but still not there, we can do better. Logistical Regression works by computing a weighted sum of the learned coefficient and the feature (aka column) value. Hence when the range of the feature values varies, features which have higher numerical values can be unduely influential in the model, affecting the accuracy of the optimisation alogrithm.

In [305]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        class_weight='balanced',  # corrects for 7:1 class imbalance
        solver='lbfgs',
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

,steps,"[('scaler', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


In [304]:
# Make predictions on test set
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # Probability of class 1

# Evaluate performance
print("Classification Report on Test Set:")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Survived', 'Dead']))

Classification Report on Test Set:
              precision    recall  f1-score   support

    Survived       0.98      0.78      0.87       359
        Dead       0.39      0.88      0.54        57

    accuracy                           0.80       416
   macro avg       0.68      0.83      0.70       416
weighted avg       0.90      0.80      0.82       416



## Conclusions
By tuning the parameters passed to the model the recall of the model has progressed from predicting 49% of deaths accurately to 88%. This has been done apparently at the expense of predicting Survived (96% to 78%) however we have argued earlier the intial number was inflated by the imbalance in the survivied vs dead data (showing that predicting survival with no model would give a recall of 86%).


### Limitations
What this analysis has shown is why a classification model such as logistic regression is inadequate for medical analysis of this nature. The binary nature of status, which was alive or dead on 31st December 2020 treats a patient who has survived 1 year from diagnosis the same as one who has survived 5. Clearly from a patient perspective the latter is a much stronger outcome. 


## To Do
- Understand the accuracy of the model
-- done this and shown with improvements, can maybe do some plots here
-- also need citations as to why we would do this.
- decide which illustrations / visuals to use and why
- look at the swtich to survival as well
- explain imputation and why it isn't relevant here
- Find the orignal paper
- Write up should contain an explaination of the data